# Random Forest Model

## 1. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from pathlib import Path
import sys
sys.path.append('../')
from src.utils import save_results

## 2. Load Data

In [2]:
print("Random Forest: Loading final pre-processed dataset...")
input_path = Path("../data/processed/final_ml_ready_dataset.csv")
results_path = "../results/model_comparison.csv"

try:
    df = pd.read_csv(input_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Dataset not found at '{input_path}'. Please run all data preparation scripts first.")

Random Forest: Loading final pre-processed dataset...
Dataset loaded successfully. Shape: (2619, 202)


## 3. Define Features (X) and Target (y)

In [3]:
target_column = 'is_fraud'
X = df.drop(columns=[target_column])
y = df[target_column]

## 4. Split Data

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 5. Define and Train Model

In [5]:
print("Random Forest: Training model...")
model_name = "Random Forest"
hyperparams = {'n_estimators': 100, 'random_state': 42, 'class_weight': 'balanced', 'max_depth': 10}
model = RandomForestClassifier(**hyperparams)

model.fit(X_train, y_train)
print("Model trained.")

Random Forest: Training model...
Model trained.


## 6. Evaluate Model

In [6]:
print("Random Forest: Evaluating model...")
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

Random Forest: Evaluating model...


## 7. Save Results

In [7]:
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1_score': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_pred_proba)
}

description = f"An ensemble of Decision Trees. Hyperparameters: {hyperparams}"

save_results(results_path, model_name, description, metrics)

Updated results for 'Random Forest' in '..\results\model_comparison.csv'.


## LLM Summary
### Findings
The Random Forest model demonstrates strong performance, outperforming the single Decision Tree as expected. Its high precision (~0.89) and solid recall (~0.67) result in a very good F1-score, indicating a strong balance between catching fraud and avoiding false alarms. The ROC AUC score is excellent (~0.96), showing superior capability in distinguishing between fraudulent and legitimate transactions across all thresholds. This confirms the power of ensemble methods on this dataset.
### Insights
For business operations, the Random Forest is a highly reliable and robust choice. Its high precision means that fraud alerts are very likely to be genuine, making the investigation process highly efficient. While it doesn't catch every fraudulent case (recall isn't perfect), it provides a significant reduction in fraud losses while maintaining customer trust by not flagging too many legitimate transactions.
### Feature Importance Interpretation
As an ensemble of many trees, the Random Forest's feature importance is more robust than a single tree's. It considers the average importance of features across all 100 trees. We expect the same top features as the Decision Tree (like `consistency_score`, `transaction_frequency`) to rank highly, but their relative importance might be distributed more evenly. This suggests that a combination of multiple weak predictors is being used effectively to create a single strong predictor.